# Phase 1 — Data audit and randomization check

Every downstream claim — ATE, Qini curves, counterfactual predictions — rests on Criteo having actually randomized treatment. Before touching any modelling, this notebook loads the raw dataset and audits the assignment mechanism.

The load-bearing check is the **balance test**. Random assignment implies treated and control arms share identical feature distributions in expectation. The **Standardized Mean Difference (SMD)** quantifies the gap on each feature; the causal-inference literature treats |SMD| < 0.05 as well-balanced. If this audit fails, everything after this notebook is built on sand.

In [ ]:
import sys
from pathlib import Path

# Find the project root (the folder containing pyproject.toml) and add it
# to sys.path so `from src.uplift.data import ...` works regardless of where
# Jupyter was launched.
here = Path.cwd()
for candidate in [here, *here.parents]:
    if (candidate / "pyproject.toml").exists():
        REPO_ROOT = candidate
        break
sys.path.insert(0, str(REPO_ROOT))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from src.uplift.data import load_criteo, FEATURE_COLS, TREATMENT_COL
from src.uplift.eda import (
    treatment_share,
    base_rates_by_arm,
    ate_diff_in_means,
    standardized_mean_differences,
    compliance_rate,
)

sns.set_theme(style="whitegrid", context="notebook", palette="deep")

## 1. Load

Snappy-compressed parquet at `data/criteo-uplift-v2.1.parquet`, written by the download script with tight dtypes (`int8` for the four binary columns, `float32` for the twelve features). Roughly 14M rows, ~700 MB in RAM.

In [ ]:
df = load_criteo()
print(f"shape:  {df.shape[0]:,} rows x {df.shape[1]} cols")
print(f"memory: {df.memory_usage(deep=True).sum() / 1e6:.1f} MB")
df.head()

## 2. Treatment allocation

Criteo split users 85/15 into treated and control. The imbalance is intentional — more information about the treated arm — and doesn't threaten validity. It does shape which estimators are appropriate: the X-learner in particular is designed for exactly this kind of split.

In [ ]:
share = treatment_share(df)
print(share.to_string())

fig, ax = plt.subplots(figsize=(5, 2))
share.plot(kind="barh", color=["#8a8880", "#b6532c"], ax=ax)
ax.set_xlabel("share of users")
ax.set_ylabel("treatment (0 = control, 1 = treated)")
ax.set_title("Treatment allocation — 85/15 split")
for p in ax.patches:
    ax.annotate(f"{p.get_width():.1%}",
                (p.get_width() + 0.01, p.get_y() + p.get_height()/2),
                va="center")
plt.tight_layout()
plt.show()

## 3. Outcome base rates by arm

If the ads have any effect, treated visits and conversions should sit above control. Conversion is a ~0.3% event — realistic for display advertising, but rare enough that later phases will need bootstrap CIs to talk about differences at all.

In [ ]:
rates = base_rates_by_arm(df, ["visit", "conversion", "exposure"])
rates.index = ["control", "treated"]
rates.style.format("{:.4%}")

## 4. Naive ATE — difference in means

Under random assignment, the raw difference in arm means identifies the average treatment effect:

$$\text{ATE} = E[Y \mid T = 1] - E[Y \mid T = 0]$$

No adjustment required — that's the whole point of randomization. The 95% CI uses a normal approximation with independent-sample variances:

$$\text{SE} = \sqrt{\frac{s^2_{\text{treated}}}{n_{\text{treated}}} + \frac{s^2_{\text{control}}}{n_{\text{control}}}}, \quad \text{95\% CI} = \text{ATE} \pm 1.96 \cdot \text{SE}$$

In [ ]:
# Using the library function
for outcome in ["visit", "conversion"]:
    ate = ate_diff_in_means(df, outcome)
    print(ate.pretty())

# Manual double-check on 'conversion' so you see the math directly
print("\nManual check on 'conversion':")
treated = df.loc[df["treatment"] == 1, "conversion"]
control = df.loc[df["treatment"] == 0, "conversion"]

diff = treated.mean() - control.mean()
se = np.sqrt(treated.var(ddof=1) / len(treated) + control.var(ddof=1) / len(control))
lo, hi = diff - 1.96 * se, diff + 1.96 * se
print(f"  diff = {diff:+.5f}, SE = {se:.6f}, 95% CI = [{lo:+.5f}, {hi:+.5f}]")

## 5. Compliance — eligibility vs. exposure

`treatment=1` marks eligibility (the randomized quantity). `exposure=1` marks whether the user actually saw an ad. Only some eligible users hit an ad-serving page, so compliance is partial.

This is **one-sided noncompliance**: controls are guaranteed `exposure=0` by construction, but only a fraction of treated users have `exposure=1`. The ATE above therefore measures **Intention-to-Treat** — the causal effect of being eligible, not of actually seeing an ad. ITT is the correct target for a targeting policy, which controls eligibility but not the browsing behavior that determines exposure.

In [ ]:
c = compliance_rate(df)
print(f"P(exposure=1 | treatment=1) = {c['p_exposed_given_treated']:.4%}")
print(f"P(exposure=1 | treatment=0) = {c['p_exposed_given_control']:.4%}")
print(f"\n-> Only ~3.6% of eligible users actually saw the ad.")

## 6. Balance test

Random assignment is independent of everything, so treated and control arms should share identical feature distributions in expectation. The **Standardized Mean Difference** measures the per-feature gap:

$$\text{SMD} = \frac{\bar{X}_{\text{treated}} - \bar{X}_{\text{control}}}{\sqrt{(s^2_{\text{treated}} + s^2_{\text{control}}) / 2}}$$

The pooled-SD denominator makes SMD scale-free — 0.1 means "arms differ by 10% of a pooled SD" regardless of the feature's units.

**Standard thresholds (Rosenbaum & Rubin; Austin 2009):**

| \|SMD\| | interpretation |
|---|---|
| < 0.05 | well balanced |
| 0.05 – 0.10 | borderline |
| ≥ 0.10 | notable imbalance — investigate |

Any feature crossing 0.10 signals broken randomization or a pipeline bug. All features under 0.05 clears causal identification.

In [ ]:
# Manual SMD computation — do this once so you internalize the math
manual_smds = []
for feat in FEATURE_COLS:
    t = df.loc[df["treatment"] == 1, feat]
    c = df.loc[df["treatment"] == 0, feat]
    pooled_sd = np.sqrt((t.var(ddof=1) + c.var(ddof=1)) / 2)
    smd = (t.mean() - c.mean()) / pooled_sd
    manual_smds.append({"feature": feat, "smd": smd, "abs_smd": abs(smd)})

manual_df = pd.DataFrame(manual_smds).sort_values("abs_smd", ascending=False).reset_index(drop=True)
manual_df.style.format({"smd": "{:+.4f}", "abs_smd": "{:.4f}"}) \
    .background_gradient(subset=["abs_smd"], cmap="Reds", vmin=0, vmax=0.10)

In [ ]:
# The 'Love plot' — standard causal-inference visualization for balance tests
smd_sorted = manual_df.sort_values("abs_smd")

fig, ax = plt.subplots(figsize=(7.5, 5))
ax.hlines(smd_sorted["feature"], 0, smd_sorted["abs_smd"], color="#4a4640", lw=1.5)
ax.scatter(smd_sorted["abs_smd"], smd_sorted["feature"], color="#b6532c", s=70, zorder=3)

ax.axvline(0.05, color="#a35a00", ls="--", lw=1.2,
           label="|SMD| = 0.05 (well-balanced threshold)")
ax.axvline(0.10, color="#9c2b2b", ls="--", lw=1.2,
           label="|SMD| = 0.10 (imbalance threshold)")

ax.set_xlim(0, 0.15)
ax.set_xlabel("|Standardized Mean Difference|")
ax.set_title("Balance test: every feature is below the 0.05 threshold")
ax.legend(loc="lower right", framealpha=1)
plt.tight_layout()
plt.show()

print(f"Worst |SMD|:            {manual_df['abs_smd'].max():.4f}")
print(f"Features |SMD| > 0.05:  {(manual_df['abs_smd'] > 0.05).sum()} / 12")
print(f"Features |SMD| > 0.10:  {(manual_df['abs_smd'] > 0.10).sum()} / 12")

## 7. Per-feature distributions

SMD compares only first moments. Two arms with matching means but different shapes pass the SMD check while carrying a real distributional imbalance. Overlaying histograms per feature catches shape mismatches — under valid randomization, the two curves should overlap on every feature.

In [ ]:
# 14M rows is too many for readable histograms — downsample to 200k
sample = df.sample(200_000, random_state=42)

fig, axes = plt.subplots(3, 4, figsize=(13, 8))
for ax, feat in zip(axes.flat, FEATURE_COLS):
    for t, color, label in [(0, "#8a8880", "control"), (1, "#b6532c", "treated")]:
        sample.loc[sample["treatment"] == t, feat].hist(
            bins=40, ax=ax, color=color, alpha=0.5, density=True, label=label
        )
    ax.set_title(feat, fontsize=10)
    ax.set_ylabel("")
    ax.grid(alpha=0.3)

axes[0, 0].legend(loc="upper right", fontsize=8)
plt.suptitle("Feature distributions by treatment arm (200k sample)", y=1.02)
plt.tight_layout()
plt.show()

## Findings

- **Dataset.** 13,979,592 rows × 16 columns, ~727 MB in RAM.
- **Allocation.** 85/15 treated/control — imbalanced by design, motivates the X-learner in Phase 3.
- **Lift on both outcomes:**
  - Visit: **+1.03 pp** [95% CI +1.01, +1.06]
  - Conversion: **+0.115 pp** [95% CI +0.108, +0.122]
- **Compliance.** 3.6% — the analysis targets Intention-to-Treat, which matches what a targeting policy can actually control.
- **Balance.** Worst |SMD| = 0.049. Zero features cross 0.05. Zero features cross 0.10. **Randomization holds.**

Causal identification is clean. Phase 2 fits the S-, T-, Class-Transformation, and X-learners plus a propensity baseline, and produces the first Qini comparison.